# 第 12 章: 正則化とモデル選択の探索と可視化

alpha を細かく変えたときの係数と決定係数の変化、ラッソ回帰で 0 になる係数の数、分け方による結果の違いを確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)
@file:DependsOn("org.tribuo:tribuo-regression-slm:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter07.r2Score
import chapter12.bestExperiment
import chapter12.fitLasso
import chapter12.fitRidge
import chapter12.prepareBoston
import chapter12.runRidgeExperiments
import chapter12.zeroCoefficientNames
import org.tribuo.regression.slm.ElasticNetCDTrainer
import java.io.File
import java.util.logging.Level
import java.util.logging.Logger
import kotlin.math.log10
import kotlin.math.pow

// Tribuo の座標降下法が収束のたびに出す INFO のログを表示しない
Logger.getLogger(ElasticNetCDTrainer::class.java.name).level = Level.WARNING

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val bostonCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "Boston.csv")
val boston = prepareBoston(bostonCsv, testSize = 0.3, validationSize = 0.3, seed = 0)
// 0.01 から 1000 までを対数の目盛りで等間隔に 30 個並べる
val alphas = (0 until 30).map { 10.0.pow(-2.0 + 5.0 * it / 29) }

## alpha と係数の変化

In [ ]:
val paths =
    alphas.flatMap { alpha ->
        val model = fitRidge(boston.xTrain, boston.tTrain, alpha)
        boston.featureNames.zip(model.coefficients) { name, coefficient -> Triple(log10(alpha), name, coefficient) }
    }
val coefficientPaths =
    dataFrameOf(
        "log10(alpha)" to paths.map { it.first },
        "特徴量" to paths.map { it.second },
        "係数" to paths.map { it.third },
    )
coefficientPaths.plot {
    line {
        x("log10(alpha)")
        y("係数")
        color("特徴量")
    }
    layout.title = "alpha と係数の変化"
}

In [ ]:
val sampleAlphas = listOf(0.01, 10.0, 1000.0)
val sampleModels = sampleAlphas.map { fitRidge(boston.xTrain, boston.tTrain, it) }
dataFrameOf(
    listOf(boston.featureNames.toColumn("特徴量")) +
        sampleAlphas.zip(sampleModels) { alpha, model -> model.coefficients.map { "%.2f".format(java.util.Locale.ROOT, it) }.toColumn("alpha=$alpha") },
)

## alpha と決定係数

In [ ]:
val experiments = runRidgeExperiments(boston.xTrain, boston.tTrain, boston.xValid, boston.tValid, alphas)
val scores =
    dataFrameOf(
        "log10(alpha)" to experiments.map { log10(it.alpha) } + experiments.map { log10(it.alpha) },
        "データ" to experiments.map { "訓練 R²" } + experiments.map { "検証 R²" },
        "決定係数" to experiments.map { it.trainScore } + experiments.map { it.validationScore },
    )
scores.plot {
    line {
        x("log10(alpha)")
        y("決定係数")
        color("データ")
    }
    layout.title = "alpha と決定係数"
}

In [ ]:
bestExperiment(experiments).let { it.alpha to it.validationScore }

## ラッソ回帰で 0 になる係数

In [ ]:
val lassoAlphas = listOf(0.01, 0.1, 0.5, 1.0, 2.0)
dataFrameOf(
    "alpha" to lassoAlphas,
    "0 になった係数の数" to lassoAlphas.map { zeroCoefficientNames(fitLasso(boston.xTrain, boston.tTrain, it).coefficients, boston.featureNames).size },
    "テスト R²" to lassoAlphas.map { r2Score(boston.tTest, fitLasso(boston.xTrain, boston.tTrain, it).predict(boston.xTest)) },
)

## 分け方による結果の違い

In [ ]:
val candidates = listOf(0.0, 0.1, 1.0, 10.0, 100.0)
val bySeed =
    (0 until 5).map { seed ->
        val d = prepareBoston(bostonCsv, testSize = 0.3, validationSize = 0.3, seed = seed)
        val best = bestExperiment(runRidgeExperiments(d.xTrain, d.tTrain, d.xValid, d.tValid, candidates))
        val linear = fitRidge(d.xTrain, d.tTrain, 0.0)
        val ridge = fitRidge(d.xTrain, d.tTrain, best.alpha)
        listOf(seed.toDouble(), best.alpha, r2Score(d.tTest, linear.predict(d.xTest)), r2Score(d.tTest, ridge.predict(d.xTest)))
    }
dataFrameOf(
    "シード" to bySeed.map { it[0].toInt() },
    "選んだ alpha" to bySeed.map { it[1] },
    "線形回帰のテスト R²" to bySeed.map { it[2] },
    "リッジ回帰のテスト R²" to bySeed.map { it[3] },
)